# Safe and idempotent methods

Two questions to ask about any request:

- **Is it safe?** Does it change nothing on the server?
- **Is it idempotent?** If I send it twice, does the server end up in the same place?

The whole lesson is those two questions, and what they mean for retries.

## Why this matters

A request can leave your program and never come back. The network drops, or the server takes too long, and the client gives up.

But the server may already have done the work.

So the client has a choice: **retry**, and risk doing it twice — or **give up**, and risk not doing it at all. Whether retrying is safe depends entirely on the method.

## Safe

A method is **safe** when the client is only asking for something, not asking for a change.

`GET /books/BK-001` is safe. Ask once, ask ten times: the book does not change.

Safe does not mean nothing happens on the server. The server may write a log line. It just means **you did not ask for a change**.

## Idempotent

A method is **idempotent** when sending it twice leaves the server in the **same state** as sending it once.

Notice: it says nothing about the reply. The second reply can be different — even an error — as long as the stored data ends up the same.

`PUT`, `DELETE` and `GET` are idempotent. `POST` is not.

## Safe is a stronger promise

Every safe method is also idempotent. Not every idempotent method is safe.

- `GET` — safe, so also idempotent.
- `DELETE` — changes things (not safe), but the end state is the same however many times you send it.

So: **safe ⊂ idempotent.**

## The table

| Method | Safe? | Idempotent? | Why |
|---|---|---|---|
| `GET` | yes | yes | only reads |
| `PUT` | no | yes | replaces with the value you sent |
| `DELETE` | no | yes | already gone the second time |
| `POST` | no | **no** | adds a new thing each time |
| `PATCH` | no | depends | depends on what the body says |

Short version: **repeat a PUT, never blindly repeat a POST.**

In [1]:
STORE = {
    "books": {"BK-001": {"title": "The Salt Road", "copies": 3}},
    "loans": [],
}


def show(label: str) -> None:
    book = STORE["books"].get("BK-001")
    print(f"{label:<22} book={book}  loans={len(STORE['loans'])}")

In [2]:
def get_book(book_id: str) -> dict:
    # Reads only. Nothing in STORE changes.
    return STORE["books"][book_id]


show("start")
get_book("BK-001")
get_book("BK-001")
get_book("BK-001")
show("after 3 GETs")

start                  book={'title': 'The Salt Road', 'copies': 3}  loans=0
after 3 GETs           book={'title': 'The Salt Road', 'copies': 3}  loans=0


Three `GET`s, no change. **Safe**, so repeating it is always fine.

In [3]:
def put_book(book_id: str, title: str, copies: int) -> int:
    # Replaces the whole book with exactly what was sent.
    STORE["books"][book_id] = {"title": title, "copies": copies}
    return 200


show("start")
put_book("BK-001", "The Salt Road", 5)
show("after 1st PUT")
put_book("BK-001", "The Salt Road", 5)
show("after 2nd PUT")

start                  book={'title': 'The Salt Road', 'copies': 3}  loans=0
after 1st PUT          book={'title': 'The Salt Road', 'copies': 5}  loans=0
after 2nd PUT          book={'title': 'The Salt Road', 'copies': 5}  loans=0


The second `PUT` writes the same values over the same values. **Same end state.**

That is why repeating a `PUT` is safe — as long as the body is **absolute**: *set copies to 5*, not *add one copy*.

In [4]:
def delete_book(book_id: str) -> int:
    if book_id not in STORE["books"]:
        return 404
    del STORE["books"][book_id]
    return 204


show("start")
print("1st DELETE ->", delete_book("BK-001"))
show("after 1st DELETE")
print("2nd DELETE ->", delete_book("BK-001"))
show("after 2nd DELETE")

start                  book={'title': 'The Salt Road', 'copies': 5}  loans=0
1st DELETE -> 204
after 1st DELETE       book=None  loans=0
2nd DELETE -> 404
after 2nd DELETE       book=None  loans=0


The replies differ: `204` then `404`. The **state is identical** — the book is gone either way.

This is the point people miss in interviews: **idempotency is about the stored state, not the status code.**

In [5]:
def post_loan(book_id: str) -> tuple[int, dict]:
    loan = {"id": len(STORE["loans"]) + 1, "book": book_id}
    STORE["loans"].append(loan)
    return 201, loan


# Put the book back: the DELETE above removed it.
STORE["books"]["BK-001"] = {"title": "The Salt Road", "copies": 3}
STORE["loans"].clear()

show("start")
print("1st POST ->", post_loan("BK-001"))
print("2nd POST ->", post_loan("BK-001"))
show("after 2 POSTs")

start                  book={'title': 'The Salt Road', 'copies': 3}  loans=0
1st POST -> (201, {'id': 1, 'book': 'BK-001'})
2nd POST -> (201, {'id': 2, 'book': 'BK-001'})
after 2 POSTs          book={'title': 'The Salt Road', 'copies': 3}  loans=2


Two loans. The member borrowed one book and now owes two.

`POST` **appends**, so each send adds another thing. Repeating it is not safe — and this is exactly what happens when a client retries after a timeout.

## The failure in real life

1. Member taps **Borrow**.
2. Server creates the loan.
3. The reply is lost on the way back.
4. The app retries.
5. The server creates a **second** loan.

The member did one thing; the system recorded two. Same with payments, orders, and any "send" button.

## Three ways to handle it

1. **Do not retry `POST` automatically.** Retry `GET`, `PUT` and `DELETE`; ask the user to retry anything else.
2. **Send an idempotency key.** A unique id per action, so the server recognises a repeat and replays the first answer.
3. **Make the action naturally idempotent.** If the request says *set state to X* instead of *add one*, repeats cost nothing.

In [10]:
SEEN_KEYS = {}

def post_loan_with_key(key: str, book_id: str) -> tuple[int, dict]:
    # The same key means "you already asked for this".
    if key in SEEN_KEYS:
        return 200, SEEN_KEYS[key]

    loan = {"id": len(STORE["loans"]) + 1, "book": book_id}
    STORE["loans"].append(loan)
    SEEN_KEYS[key] = loan
    return 201, loan

STORE["loans"] = []
show("start")
print("1st ->", post_loan_with_key("req-001", "BK-001"))
print("2nd ->", post_loan_with_key("req-001", "BK-001"))
print("new key ->", post_loan_with_key("req-002", "BK-001"))
show("after 3 POSTs")

start                  book={'title': 'The Salt Road', 'copies': 4}  loans=0
1st -> (201, {'id': 1, 'book': 'BK-001'})
2nd -> (200, {'id': 1, 'book': 'BK-001'})
new key -> (201, {'id': 2, 'book': 'BK-001'})
after 3 POSTs          book={'title': 'The Salt Road', 'copies': 4}  loans=2


Same key twice: **one** loan, and the first receipt is replayed. A new key means a genuinely new action, so it goes through.

Keep the keys in a table with a time limit — an hour, a day — and delete old ones.

## PUT replaces, PATCH changes

`PUT` sends the **whole resource**, so repeats land on the same values.

`PATCH` sends **only the change**, and the change can be relative — *add one copy*. Send that twice and you have added two.

`PATCH` is not automatically wrong. It is just **not automatically idempotent**. It depends on the body.

In [7]:
def patch_add_copies(book_id: str, n: int) -> None:
    # Relative change: "add n copies".
    STORE["books"][book_id]["copies"] += n


def patch_set_copies(book_id: str, n: int) -> None:
    # Absolute change: "copies should be n".
    STORE["books"][book_id]["copies"] = n


STORE["books"]["BK-001"]["copies"] = 3
patch_add_copies("BK-001", 1)
patch_add_copies("BK-001", 1)
show("after 2x add 1")

STORE["books"]["BK-001"]["copies"] = 3
patch_set_copies("BK-001", 4)
patch_set_copies("BK-001", 4)
show("after 2x set 4")

after 2x add 1         book={'title': 'The Salt Road', 'copies': 5}  loans=2
after 2x set 4         book={'title': 'The Salt Road', 'copies': 4}  loans=2


Same verb, two outcomes. `add 1` twice gives 5; `set 4` twice gives 4.

So the rule is not about the verb alone. **Absolute requests are safe to repeat. Relative requests are not.**

## Status codes that trip people up

| Situation | Status |
|---|---|
| First `POST` creates a loan | `201 Created` |
| Repeat with the same idempotency key | `200 OK` with the original body |
| `DELETE` succeeds | `204 No Content` |
| `DELETE` again, already gone | `404 Not Found` — still idempotent |
| `POST` conflicts with an existing unique thing | `409 Conflict` |

The `404` on the second `DELETE` is correct. Nothing is broken.

In [12]:
def borrow_once(book_id: str, member: str) -> tuple[int, str]:
    # One open loan per member per book: duplicates are a conflict, not a new loan.
    for loan in STORE["loans"]:
        if loan["book"] == book_id and loan["member"] == member:
            return 409, "This member already has this book out."

    STORE["loans"].append({"id": len(STORE["loans"]) + 1, "book": book_id, "member": member})
    return 201, "Loan created."


STORE["loans"] = []
print(borrow_once("BK-001", "Ada"))
print(borrow_once("BK-001", "Ada"))
print("loans:", len(STORE["loans"]))

(201, 'Loan created.')
(409, 'This member already has this book out.')
loans: 1


A unique rule in the data turns a dangerous repeat into a safe `409`. The second request changes nothing.

This is the cheapest protection there is: let the database refuse the duplicate.

## Which verb for the library API

| Endpoint | Method | Safe | Idempotent |
|---|---|---|---|
| `/books` | `GET` | yes | yes |
| `/books/BK-001` | `GET` | yes | yes |
| `/books/BK-001` | `PUT` | no | yes |
| `/books/BK-001` | `DELETE` | no | yes |
| `/loans` | `POST` | no | **no** |

Only `/loans` needs care on retry: an idempotency key, or a one-open-loan rule.

In [16]:
import hashlib

INGESTED = {}


def ingest_document(text: str) -> tuple[str, str]:
    # The same text always hashes the same, so a repeat is easy to spot.
    digest = hashlib.sha256(text.encode()).hexdigest()[:12]
    if digest in INGESTED:
        return "skipped", INGESTED[digest]

    doc_id = f"DOC-{len(INGESTED) + 1:03d}"
    INGESTED[digest] = doc_id
    return "ingested", doc_id


page = "Refunds take five working days."
print(ingest_document(page))
print(ingest_document(page))
print(ingest_document("Late returns are fined 50 naira a day."))

('ingested', 'DOC-001')
('skipped', 'DOC-001')
('ingested', 'DOC-002')


That is the Harbour version of the same idea: an upload or ingestion that runs twice must not create two copies of the same page.

The content hash is the key. Same bytes, same hash, already stored — skip it. This is what makes a re-run safe in weeks 8 and 9.

## Summary

| Method | Safe | Idempotent | Retry freely? |
|---|---|---|---|
| `GET` | yes | yes | yes |
| `PUT` | no | yes | yes |
| `DELETE` | no | yes | yes |
| `POST` | no | no | only with a key or a unique rule |
| `PATCH` | no | depends on the body | check the body first |

`HEAD` and `OPTIONS` are safe and idempotent too. `TRACE` as well, but you will rarely write it.

## What to carry away

- **Safe** means the client asked for no change.
- **Idempotent** means repeats leave the same stored state, even if the replies differ.
- `DELETE` twice gives `204` then `404` and is still idempotent.
- `POST` twice gives two things. That is the one to protect.
- Protect it with an **idempotency key** or a **unique rule** in the data.
- **Absolute requests repeat safely. Relative requests do not.**

One line: **repeat reads and replacements; guard anything that adds.**